# 📐 S&P 500 Pairs Trading 形成期 (Formation Period) 策略邏輯與公式詳解

## 📝 概述

在配對交易 (Pairs Trading) 中，**形成期 (Formation Period)**（預設 $F = 252$ 天）的核心任務是**篩選候選標的並建立具備統計套利價值的配對組合**。

> **⚠ 注意：** 本文件以 `strategies/formation/` 下實際運行的 `.py` 原始碼為唯一依據，所有公式、閾值與欄位命名均與程式碼保持一致。

#### 目前啟用策略（`config.py` `strategies_raw = strategies_raw_all[:]`，共 15 個）

| 編號 | 策略名稱 | 形成期模組 | 交易期模組 |
| :---: | :--- | :--- | :--- |
| 1 | SSD Basic | `ssd_basic.py` | `zscore_trading.py` |
| 2 | SSD Rolling | `ssd_rolling.py` | `zscore_trading.py` |
| 3 | DTW Paper (DTW) | `DTW_Cointegration_Paper.py` | `zscore_trading.py` |
| 4 | DTW Paper (SSD-DTW-PCA) | `DTW_Cointegration_Paper.py` | `zscore_trading.py` |
| 5 | HDBSCAN MultiScale | `HDBSCAN_MultiScale.py` | `zscore_trading.py` |
| 6 | HDBSCAN MultiScale PCA-UMAP | `HDBSCAN_MultiScale.py` | `zscore_trading.py` |
| 7 | HDBSCAN UMAP | `HDBSCAN_UMAP.py` | `zscore_trading.py` |
| 8 | HDBSCAN UMAP PCA-UMAP | `HDBSCAN_UMAP.py` | `zscore_trading.py` |
| 9 | Ensemble HDBSCAN | `ensemble.py` | `zscore_trading.py` |
| 10 | Ensemble SSD-DTW | `ensemble.py` | `zscore_trading.py` |
| 11 | SSD Rolling DRL | `ssd_rolling.py` | `drl_lstm_trading.py` |
| 12 | HDBSCAN UMAP DRL | `HDBSCAN_UMAP.py` | `drl_lstm_trading.py` |
| 13 | HDBSCAN MultiScale DRL | `HDBSCAN_MultiScale.py` | `drl_lstm_trading.py` |
| 14 | SSD Rolling Kalman | `ssd_rolling.py` | `kalman_trading.py` |
| 15 | HDBSCAN UMAP Kalman | `HDBSCAN_UMAP.py` | `kalman_trading.py` |

#### 📂 所有形成期策略檔案

- `ssd_basic.py` — 累積回報指數 SSD，等權對沖
- `ssd_rolling.py` — 對數價格 Z-Score SSD，OLS 對沖比例
- `DTW_Cointegration_Paper.py` — 共整合篩選 + Sakoe-Chiba DTW + PCA 融合
- `HDBSCAN_UMAP.py` — 密度分群 + 10 維單股特徵 + 10 維配對品質評分
- `HDBSCAN_MultiScale.py` — 密度分群 + 多時間尺度跨期穩定性評分
- `ensemble.py` — 雙子策略交集優先集成

---

#### 統一三道統計過濾（所有策略共用）

| 過濾條件 | 公式 / 判定 | SSD | DTW | HDBSCAN UMAP | HDBSCAN MultiScale |
| :--- | :--- | :---: | :---: | :---: | :---: |
| ADF 共整合 p 值上限 | $p <$ threshold | 0.05 | 0.01 | 0.01 | 0.05 |
| OU 半衰期範圍 | $1 \le HL \le 42$ 天 | ✓ | ✓ | ✓ | ✓ |
| Hurst 指數上限 | $H < 0.50$ | ✓ | ✓ | ✓ | ✓ |

> **ADF 使用 `regression="n"`**（無截距無趨勢），相比 Engle-Granger 標準程序（`"c"`）更保守。
> **OU 半衰期** 計算：$\lambda$ 為 AR(1) 係數，$HL = -\ln(2)/\lambda$，$\lambda < 0$ 才具均值回歸性。
> **Hurst 指數** 使用 R/S 分析，對殘差序列直接計算（`already_stationary=True`，不再做一次差分）。

## 📏 一、 SSD Basic 與 SSD Rolling 形成期邏輯

### 1.1 經典 SSD Basic (`ssd_basic.py`) — 累積回報指數等權對沖

**步驟 1：累積回報指數正規化**（對應 Gatev et al. 2006）

$$P'_{i,t} = \frac{P_{i,t}}{P_{i,0}}$$

以形成期第一日價格 $P_{i,0}$ 為基準，起點為 1.0，保留股票間的價格絕對水準差異。

**步驟 2：同產業 SSD 距離**

$$\text{SSD}_{A,B} = \sum_{t=1}^{F} (P'_{A,t} - P'_{B,t})^2$$

僅限同 GICS 產業內的股票對計算，避免行業結構性差異干擾。

**步驟 3：固定避險比例 $\beta = 1.0$**（等市值對沖，美元中性）

殘差定義：$\epsilon_t = P'_{A,t} - 1.0 \cdot P'_{B,t}$

**三道統計過濾**（優化：先按 SSD 排序取前 $\max(200, 15N)$ 候選對再做慢速統計計算）：
1. ADF 共整合：$p < 0.05$
2. OU 半衰期：$1.0 \le HL \le 60.0$ 天
3. Hurst 指數：$H < 0.50$（對殘差 $\epsilon_t$ 直接計算，`already_stationary=True`）

**存入 `Formation_Params` 的欄位：**

| 欄位 | 說明 |
| :--- | :--- |
| `Hedge_Ratio` | 固定 1.0 |
| `Spread_Mean` | $\mu_\epsilon = \mathbb{E}[\epsilon_t]$ |
| `Spread_Std` | $\sigma_\epsilon$ |
| `First_Price_A` / `First_Price_B` | 形成期首日原始價格（交易期正規化基準） |

---

### 1.2 OLS SSD Rolling (`ssd_rolling.py`) — Z-Score 標準化 OLS 對沖

**步驟 1：對數價格 Z-Score 標準化**

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_{\ln P_i}}{\sigma_{\ln P_i}}$$

使不同絕對價格水準的股票具有相同的波動度尺度。

**步驟 2：協方差矩陣 OLS 避險比例**（批次計算，`外層 i = Ticker_B = X，內層 j = Ticker_A = Y`）

$$\beta = \frac{\text{Cov}(P'_A,\, P'_B)}{\text{Var}(P'_B)}, \quad \epsilon_t = P'_{A,t} - \beta \cdot P'_{B,t}$$

**步驟 3：三道統計過濾**（同 SSD Basic，閾值相同）

**存入 `Formation_Params` 的欄位：**

| 欄位 | 說明 |
| :--- | :--- |
| `Hedge_Ratio` | 形成期 OLS 斜率 $\beta$ |
| `Spread_Mean` | $\mu_\epsilon$ |
| `Spread_Std` | $\sigma_\epsilon$ |
| `Log_Mean_A` / `Log_Mean_B` | 形成期對數價格均值（Z-Score 中心化用） |
| `Log_Std_A` / `Log_Std_B` | 形成期對數價格標準差（Z-Score 縮放用） |

> **SSD Basic vs Rolling 核心差異**：Basic 用累積回報指數（價格比值空間），Rolling 用 Z-Score 對數空間；Basic 固定 $\beta=1$，Rolling 估計 OLS $\beta$；存儲的參考基準也不同（`First_Price` vs `Log_Mean/Std`），交易期 spread 重建路徑因此分叉。

## ⏳ 二、 DTW Cointegration Paper 形成期邏輯 (`DTW_Cointegration_Paper.py`)

動態時間扭曲（DTW）容忍兩支股票走勢存在**時間滯後**（如一隻領先一隻落後）的情況，比純 SSD 歐氏距離更具彈性。

### 步驟 1：對數價格 Z-Score 標準化（同 ssd_rolling）

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_{\ln P_i}}{\sigma_{\ln P_i}}$$

### 步驟 2：雙向 OLS 共整合篩選（取 ADF p 值較小的方向）

對每對股票分別做兩個方向的 OLS 回歸：

$$\text{方向 AB}: \quad y = \alpha_{AB} + \beta_{AB} x + \epsilon_{AB}$$
$$\text{方向 BA}: \quad x = \alpha_{BA} + \beta_{BA} y + \epsilon_{BA}$$

選擇 ADF p 值較小（共整合性更強）的方向作為 $(\text{Ticker\_A},\, \text{Ticker\_B})$，確保 Y 變數是共整合性更強的那隻。

**四道統計過濾**（$p < 0.01$）：
1. ADF 共整合：$p < 0.01$（比 SSD 更嚴格）
2. OU 半衰期：$1.0 \le HL \le 60.0$ 天
3. Hurst 指數：$H < 0.50$（對 OLS 殘差直接計算）
4. 篩選後才計算 SSD/DTW 距離（節省計算）

### 步驟 3：Sakoe-Chiba 限制窗口 DTW 距離

$$\text{DTW}_{A,B} = \min_{\text{path}} \sum_{(i,j) \in \text{path}} (P'_{A,i} - P'_{B,j})^2, \quad |i-j| \le W$$

限制時間扭曲在 $W=15$ 天內，防止非理性的長距離時間對齊。

### 步驟 4：排序模式

| 模式 | 說明 |
| :--- | :--- |
| `method="dtw"` | 依 DTW 距離升序排序（對照組） |
| `method="ssd_dtw_pca"` | SSD 與 DTW 標準化後 PCA 取第一主成分升序排序（實驗組）；若 PC1 loadings 方向為負則取反確保「距離越小得分越小」 |

### 存入 `Formation_Params` 的欄位：

| 欄位 | 說明 |
| :--- | :--- |
| `Hedge_Ratio` | 最佳方向 OLS 斜率 $\beta$ |
| `OLS_Alpha` | 最佳方向 OLS 截距 $\alpha$（交易期重建 spread 必需） |
| `Spread_Mean` | $\mu_\epsilon$（OLS 殘差均值，因含截距 $\approx 0$） |
| `Spread_Std` | $\sigma_\epsilon$ |
| `Log_Mean_A` / `Log_Mean_B` | 形成期對數價格均值 |
| `Log_Std_A` / `Log_Std_B` | 形成期對數價格標準差 |

> **與 SSD Rolling 的關鍵差異**：DTW 存入 `OLS_Alpha`，使交易期可以在原始對數價格空間重建 spread（$\text{Spread} = \ln P_A - \alpha - \beta \ln P_B$），而非在 Z-Score 標準化空間計算。這是交易期 zscore_trading.py **路徑 A** 的觸發條件。

## 🌐 三、 HDBSCAN UMAP 形成期邏輯 (`HDBSCAN_UMAP.py`)

HDBSCAN UMAP 採用「**深度**」策略：用完整形成期的 10 維單股特徵描述每支股票行為，再在聚類群落內計算 10 維配對品質，選出均值回歸特性最強的配對。

### 3.1 第一層：單股特徵萃取（10 維）

| 特徵 | 說明 |
| :--- | :--- |
| `vol_21`, `vol_63`, `vol_all` | 近 21/63 日和整期收益率波動率 |
| `vol_ratio_short_long` | $\text{vol}_{21} / \text{vol}_{63}$（近期 vs 中期波動率比） |
| `autocorr_1` | 日收益率 1 階自相關係數 |
| `skew_all`, `kurt_all` | 收益率偏態與峰態 |
| `hurst` | R/S Hurst 指數（對原始對數價格序列，`already_stationary=False`） |
| `beta` | 對市場代理（等權報酬）的 Beta |
| `log_price_level` | 形成期對數價格均值 |

**降維與聚類**：RobustScaler 標準化 → UMAP 降至 5 維（預設）→ HDBSCAN 分群（`min_cluster_size=5`，有重試機制逐步降低到 2）。噪音點（`label=-1`）直接排除，不參與配對。

### 3.2 第二層：配對品質評分（10 維）

| 特徵 | 計算方式 |
| :--- | :--- |
| `hurst_spread` | 對 OLS 殘差序列計算 Hurst（`already_stationary=True`） |
| `halflife` | OU 半衰期 |
| `zero_crossings` | 殘差穿越均值次數 |
| `corr_mean`, `corr_stability` | 滾動 60 日 Pearson 相關均值與標準差 |
| `corr_regime_diff` | 牛市/熊市相關係數差值（市場狀態穩定性） |
| `beta_diff` | 兩股票 Beta 差的絕對值 |
| `vol_ratio` | 兩股票波動率比（$\max/\min$） |
| `volume_corr` | 成交量對數相關（缺乏成交量資料時為 0.0） |
| `adv_ratio` | 平均成交量比（缺乏成交量資料時為 1.0） |

### 3.3 品質評分公式（各分項 $\in [0,1]$）

$$\text{Quality\_Score} = \sum_{k} w_k \cdot s_k$$

| 分項 $s_k$ | 計算 | 權重 $w_k$ |
| :--- | :--- | :---: |
| $s_\text{hurst}$ | $\max(0,\, (0.5 - H) / 0.5)$ | 0.20 |
| $s_\text{halflife}$ | 1.0 若 $5 \le HL \le 30$；線性插值若 $HL<5$ 或 $30<HL\le 60$；0 若 $>60$ | 0.15 |
| $s_\text{zc}$ | $\min(1,\, \text{zero\_crossings} / 200)$ | 0.10 |
| $s_\text{corr\_mean}$ | $\max(0,\, (\bar{\rho} - 0.5) / 0.5)$ | 0.15 |
| $s_\text{corr\_stab}$ | $\max(0,\, 1 - \sigma_\rho / 0.3)$ | 0.15 |
| $s_\text{regime}$ | $\max(0,\, 1 - \Delta\rho / 0.4)$ | 0.05 |
| $s_\text{beta}$ | $\max(0,\, 1 - |\Delta\beta| / 1.0)$ | 0.08 |
| $s_\text{vol}$ | $\max(0,\, 1 - (r_\sigma - 1) / 2)$ | 0.05 |
| $s_\text{vol\_corr}$ | $\max(0,\, \rho_\text{vol})$（或 0 若無資料） | 0.04 |
| $s_\text{adv}$ | $\text{adv\_ratio}$ | 0.03 |

### 3.4 額外過濾（HDBSCAN UMAP 特有）

除三道統計過濾外，額外有：
- **最小相關性**：$\bar{\rho} \ge 0.50$（`min_corr`）
- **最小穿越次數**：$\ge 5$（`min_zero_crossings`）
- **Mom1 截面動量差**（可選）：排除近期動量方向相同的配對

### 存入 `Formation_Params` 的欄位：

| 欄位 | 說明 |
| :--- | :--- |
| `Hedge_Ratio` | 形成期 OLS $\beta$ |
| `OLS_Alpha` | 形成期 OLS $\alpha$（原始對數價格空間） |
| `Spread_Mean` | $\mu_\epsilon$（原始 log-price OLS 殘差均值 $\approx 0$） |
| `Spread_Std` | $\sigma_\epsilon$ |
| `Log_Mean_A/B`, `Log_Std_A/B` | 形成期對數價格均值/標準差 |
| `Quality_Score` | 加權品質評分 |

## 🕐 四、 HDBSCAN MultiScale 形成期邏輯 (`HDBSCAN_MultiScale.py`)

HDBSCAN MultiScale 採用「**廣度**」策略：不求在單一時期最優，而是篩選在**多個市場週期內持續穩定**的配對，衡量跨週期的共整合一致性。

### 4.1 子時期自適應切分

不使用固定日曆邊界，而是**以當次形成期的起訖日為基準**，等比例切割為 `n_splits=4` 段（確保每個 252 天滾動窗口都有 4 段可計算）：

| 子期間 | 長度 | 說明 |
| :---: | :---: | :--- |
| Q1 | ¼ 形成期 | 最早 ~63 天 |
| Q2 | ¼ 形成期 | 次早 ~63 天 |
| Q3 | ¼ 形成期 | 次晚 ~63 天 |
| Q4 | ¼ 形成期 | 最晚 ~63 天 |

> 無預知未來問題：所有子期間均在形成期窗口內，不跨入交易期。

### 4.2 動態熊市判斷

不使用硬編碼歷史日期，而是即時計算形成期內每日的市場狀態：

$$\text{Bear}_t = \left(\exp\!\left(\frac{1}{N}\sum_i \ln P_{i,t}\right) < \text{MA}_{60}\right)$$

等權 log-price 指數低於 60 日移動平均即判定為熊市日，動態標記 `bear_mask`。

### 4.3 單股特徵萃取（多時間尺度）

對每支股票，在各子期間計算滾動特徵統計（均值、標準差、最差值、牛熊差等），拼接成跨時期的多維特徵向量。之後同樣透過 RobustScaler + UMAP/PCA 降維 + HDBSCAN 聚類。

### 4.4 多時間尺度品質評分

| 評分分項 | 說明 | 權重 |
| :--- | :--- | :---: |
| `corr_mean` 跨期均值 | 各子期間相關性均值 | 0.18 |
| `coint_pass_rate` | 子期間通過共整合檢定比例 | 0.15 |
| `hurst_mean` | 各子期間 Hurst 指數均值 | 0.12 |
| `corr_std` | 子期間相關性標準差（越小越穩） | 0.10 |
| `coint_worst_pval` | 最差子期間的 ADF p 值 | 0.10 |
| `corr_min` | 最差子期間相關係數 | 0.08 |
| `hurst_worst` | 最差子期間 Hurst | 0.08 |
| `regime_diff` | 牛熊相關差值（動態計算） | 0.06 |
| `vol_ratio_mean` | 波動率比均值 | 0.04 |
| `vol_ratio_std` | 波動率比標準差 | 0.03 |
| `Mom1_Diff` | 截面動量差（Han et al. 2021，可選） | 0.03 |
| 其他輔助項 | corr_bear 等 | 0.03 |

Coverage 評分：$s_{\text{coverage}} = \min\!\left(1.0,\, n_{\text{valid\_periods}} / n_{\text{splits}}\right)$，分母與實際子期間數一致，滿分可達。

> **與 HDBSCAN UMAP 核心差異**：UMAP 版用「完整形成期」的深度特徵（10 維配對品質）；MultiScale 版用「切片跨期」的廣度特徵（多週期穩定性）。兩者互補，故 ensemble.py 支援將二者取交集。

### 存入 `Formation_Params` 的欄位：

與 HDBSCAN UMAP 完全相同：`Hedge_Ratio`、`OLS_Alpha`、`Spread_Mean`、`Spread_Std`、`Log_Mean_A/B`、`Log_Std_A/B`、`Quality_Score`。

---

## 🔗 五、 Ensemble 集成策略 (`ensemble.py`)

Ensemble 策略不執行任何統計篩選，而是**協調兩個子策略的選出結果**，以提高配對篩選的確定性。

### 5.1 排名邏輯

1. 各子策略各自用更大的候選池（`top_n × sub_top_n_multiplier`，預設 3 倍）執行篩選
2. **正則化配對 key**：`(min(A,B), max(A,B))` 確保 (A,B) 與 (B,A) 視為同一配對
3. **Tier 1（交集）**：兩個子策略均選出的配對，依**平均排名**升序排列（確定性最高）
4. **Tier 2（補足）**：僅一個子策略選出的配對，依**最小排名**升序補至 `top_n`

### 5.2 方向正則化（確保 Ticker_A = min 字母序）

當某配對的原始子策略方向與正則 key 不同，需翻轉 OLS 相關欄位：

$$\beta_{\text{new}} = \frac{1}{\beta}, \quad \alpha_{\text{new}} = \frac{-\alpha}{\beta}$$

並同步交換 `Sector_A/B`、`Log_Mean_A/B`、`Log_Std_A/B`、`First_Price_A/B`。

### 5.3 配對資料選擇（同配對被兩個子策略都選出時）

若兩個子策略均選出同一配對（Tier 1 情況），優先保留 `Quality_Score` 較高者的欄位；若 `Quality_Score` 相同，則以 `Pair_Rank` 較小者優先。

In [ ]:
"""
HDBSCAN_UMAP.py — 品質評分計算示例

完整重現 _pair_quality_score() 的邏輯，用於驗證分項分數計算。
"""
import numpy as np

def pair_quality_score(hurst_spread, halflife, zero_crossings,
                       corr_mean, corr_stability, corr_regime_diff,
                       beta_diff, vol_ratio, volume_corr, adv_ratio):
    # 各分項分數 [0, 1]
    s_hurst = max(0.0, (0.5 - hurst_spread) / 0.5)

    if 5 <= halflife <= 30:
        s_halflife = 1.0
    elif halflife < 5:
        s_halflife = halflife / 5.0
    elif halflife <= 60:
        s_halflife = (60 - halflife) / 30.0
    else:
        s_halflife = 0.0

    s_zc        = min(1.0, zero_crossings / 200.0)
    s_corr_mean = max(0.0, (corr_mean - 0.5) / 0.5)
    s_corr_stab = max(0.0, 1.0 - corr_stability / 0.3)
    s_regime    = max(0.0, 1.0 - corr_regime_diff / 0.4)
    s_beta      = max(0.0, 1.0 - beta_diff / 1.0)
    s_vol       = max(0.0, 1.0 - (vol_ratio - 1.0) / 2.0)
    s_vol_corr  = max(0.0, volume_corr) if np.isfinite(volume_corr) else 0.0
    s_adv       = adv_ratio

    score = (0.20 * s_hurst + 0.15 * s_halflife + 0.10 * s_zc +
             0.15 * s_corr_mean + 0.15 * s_corr_stab + 0.05 * s_regime +
             0.08 * s_beta + 0.05 * s_vol + 0.04 * s_vol_corr + 0.03 * s_adv)
    return score

# 示例：一個良好的均值回歸配對
example_score = pair_quality_score(
    hurst_spread=0.30,   # H < 0.5 → 均值回歸
    halflife=15,          # 15 日半衰期 → 滿分
    zero_crossings=45,    # 穿越 45 次 / 200 = 0.225
    corr_mean=0.75,       # 相關性好
    corr_stability=0.10,  # 很穩定
    corr_regime_diff=0.15,
    beta_diff=0.20,
    vol_ratio=1.3,
    volume_corr=0.0,      # 無成交量資料 → 0
    adv_ratio=1.0         # 無成交量資料 → 1
)
print(f"示例品質評分：{example_score:.4f}")

## 📊 六、 所有策略形成期對比總表

### 6.1 篩選機制與輸出欄位對比

| 特徵 | SSD Basic | SSD Rolling | DTW Paper | HDBSCAN UMAP | HDBSCAN MultiScale |
| :--- | :---: | :---: | :---: | :---: | :---: |
| 檔案 | `ssd_basic` | `ssd_rolling` | `DTW_Paper` | `HDBSCAN_UMAP` | `HDBSCAN_MultiScale` |
| 跨產業配對 | ✗ | ✗ | ✗ | ✓ | ✓ |
| 正規化空間 | $P/P_0$ | Z-Score log | Z-Score log | Z-Score log | Z-Score log |
| 降維/聚類 | 無 | 無 | 無 | UMAP+HDBSCAN | UMAP+HDBSCAN |
| ADF 閾值 | 0.05 | 0.05 | **0.01** | **0.01** | 0.05 |
| OU 半衰期範圍 | 1–42 天 | 1–42 天 | 1–42 天 | 1–42 天 | 1–42 天 |
| Hurst 閾值 | < 0.50 | < 0.50 | < 0.50 | < 0.50 | < 0.50 |
| 避險比例 $\beta$ | 固定 1.0 | OLS（Z-Score 空間） | OLS（Z-Score 空間，雙向選最佳） | OLS（log-price 空間） | OLS（log-price 空間） |
| 存 `OLS_Alpha` | ✗ | ✗ | **✓** | **✓** | **✓** |
| 存 `First_Price_A/B` | **✓** | ✗ | ✗ | ✗ | ✗ |
| 存 `Log_Mean/Std_A/B` | ✗ | **✓** | **✓** | **✓** | **✓** |
| 品質評分排序 | SSD↑ | SSD↑ | DTW↑ / PC1↑ | Quality_Score↓ | Quality_Score↓ |
| 目前啟用 | ✗ | **✓**（Kalman） | ✗ | **✓**（Kalman） | **✓**（DRL） |

### 6.2 交易期 Spread 重建路徑（由 `Formation_Params` 決定）

形成期存的欄位決定了交易期 `zscore_trading.py` 走哪條路徑：

```
有 OLS_Alpha（不為 None）→ 路徑 A（log-price OLS 殘差空間）
  spread = ln(P_A) - alpha - beta * ln(P_B)
  → 適用於 DTW、HDBSCAN UMAP、HDBSCAN MultiScale

無 OLS_Alpha + 有 First_Price_A/B → 路徑 B1（累積回報比值空間）
  spread = P_A / P_A0 - P_B / P_B0
  → 適用於 SSD Basic

無 OLS_Alpha + 無 First_Price_A/B → 路徑 B2（Z-Score 標準化對數空間）
  spread = Z-Score(ln P_A) - beta * Z-Score(ln P_B)
  → 適用於 SSD Rolling
```

> 這三條路徑保證交易期 spread 的空間與形成期計算 `Spread_Mean`/`Spread_Std` 的空間完全一致，避免 Z-Score 量綱錯誤。

## 📖 參考文獻

本研究各形成期策略直接引自以下論著：

---

### Gatev, Goetzmann & Rouwenhorst (2006) — SSD Basic 策略基礎

> **論著：** Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative value arbitrage rule. *Review of Financial Studies*, **19**(3), 797–827.

**論著核心貢獻：**

- 提出以**累積回報指數（normalized price index）** $P'_{i,t} = P_{i,t}/P_{i,0}$ 作為配對距離計算基礎，消除絕對價格差距
- 採用**歐氏距離平方和（SSD）**作為相似度量度，計算直觀且效率高
- 固定等市值對沖比例（$\beta = 1.0$），實現美元中性策略，消除市場 Beta 暴露
- Z-Score 突破 $\pm 2\sigma$ 進場、回歸均值出場作為基準交易訊號
- 1962–2002 年 S&P 500 資料，年化超額報酬約 **6%**（交易成本前），驗證統計套利可行性

**本研究對應：** `ssd_basic.py`（SSD Basic 形成期）

---

### Krauss, Do & Huck (2016) — 三道統計過濾的學術依據

> **論著：** Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies: Distance, cointegration and copula methods. *European Journal of Operational Research*.

**論著核心貢獻：**

- 系統比較三類配對選取方法：**距離法（SSD）**、**共整合法（Engle-Granger）**、**Copula 法**
- 共整合法在風險調整後報酬上優於純距離法，因其捕捉長期均衡關係
- 引入 **ADF 單根檢定**驗證殘差定常性（stationarity），篩選真正共整合的配對
- **Hurst 指數**（$H < 0.5$）輔助過濾，驗證殘差具均值回歸性（antipersistent）
- **OU 半衰期**確保交易期內有足夠的均值回歸次數

**本研究對應：** 三道統計過濾（所有形成期策略共用）：ADF $p < 0.01$–$0.05$、$1 \le HL \le 42$、$H < 0.50$

---

### Zhu (2024) — DTW 形成期策略來源

> **論著：** Zhu, M. (2024). Pairs trading with dynamic time warping. Working Paper.

**論著核心貢獻：**

- 引入**動態時間扭曲（DTW）**度量股票走勢相似度，容忍時間滯後（lead-lag effect）
- Sakoe-Chiba 帶約束（$W=15$ 天）防止不合理的長距離時間對齊
- **雙向 OLS 共整合篩選**：同時計算 AB 與 BA 方向，取 ADF $p$ 值較小者為主方向
- SSD 與 DTW 距離的 PCA 融合（`ssd_dtw_pca`）取第一主成分作為融合排序依據
- OLS 截距 $\alpha$ 顯式保存，交易期在原始 log-price 空間重建 Spread，避免量綱錯誤

**本研究對應：** `DTW_Cointegration_Paper.py`

---

### Han, He, Rapach & Zhou (2021) — 截面動量過濾（Mom1_Diff）

> **論著：** Han, Y., He, A., Rapach, D., & Zhou, G. (2021). In search of pairs using firm fundamentals. Working Paper.

**論著核心貢獻：**

- 以**企業基本面特徵**（財務比率、盈利能力等）而非純價格序列識別潛在配對
- 提出**截面動量差（Mom1_Diff）**：排除近期動量方向相同的配對，避免追漲追跌式進場
- 基本面驅動的配對策略在嚴格交易成本假設下仍保有顯著超額報酬

**本研究對應：** `Mom1_Diff` 動量過濾（HDBSCAN MultiScale 可選項）

---

### Campello, Moulavi & Sander (2013) — HDBSCAN 聚類演算法

> **論著：** Campello, R. J., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD*.

**論著核心貢獻：**

- 提出**層次密度分群（HDBSCAN）**：自動確定群數，無需預先指定 $k$
- 噪音點（outliers）標記為 `label=-1`，適合過濾行為異常的股票
- 對群形狀和密度無嚴格假設，比 K-Means 更適合金融資料的不規則分布

**本研究對應：** `HDBSCAN_UMAP.py`、`HDBSCAN_MultiScale.py`

---

### McInnes, Healy & Melville (2018) — UMAP 降維

> **論著：** McInnes, L., Healy, J., & Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction. *arXiv:1802.03426*.

**論著核心貢獻：**

- 基於黎曼幾何與代數拓撲，保留數據的局部與全局流形結構
- 比 t-SNE 更快速，且在低維嵌入中更好保留全局結構，適合後續聚類分析
- 本研究將 10 維單股特徵降至 5 維，再輸入 HDBSCAN 聚類

**本研究對應：** `HDBSCAN_UMAP.py`、`HDBSCAN_MultiScale.py`

---

### Engle & Granger (1987) — 共整合統計理論

> **論著：** Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction: Representation, estimation, and testing. *Econometrica*, **55**(2), 251–276.

**論著核心貢獻：**

- 提出**共整合（cointegration）**的正式定義：兩個 I(1) 序列的線性組合為 I(0) 則稱共整合
- ADF 單根檢定作為共整合殘差定常性的標準驗證方法
- **誤差修正模型（ECM）**：描述短期動態與長期均衡的關係

**本研究對應：** 所有策略的 ADF 共整合過濾